# Modelado Predictivo e Inferencia (MiBici)

Este notebook orquesta la ejecución del pipeline modularizado de modelado (`mibici.modeling.train`), el registro en MLflow (`mibici.modeling.tracking`) y la inferencia con el modelo cargado desde el Model Registry (`mibici.modeling.predict`), consumiendo la configuración de `params.yaml`.

In [1]:
import mlflow
import pandas as pd
from sklearn.linear_model import Ridge

from mibici.config import PROCESSED_DATA_DIR, cargar_parametros
from mibici.modeling.predict import predecir
from mibici.modeling.tracking import registrar_modelo
from mibici.modeling.train import entrenar_y_evaluar

2026-11-05 12:00:19.422 | INFO     | mibici.config:<module>:11 - PROJ_ROOT path is: /Users/arturoflores/Library/CloudStorage/GoogleDrive-floresamaroarturo@gmail.com/My Drive/Tec de Monterrey/MLOps/mibici_project/mibici-demand


## 1. Carga de Configuración y Dataset v3

In [2]:
params = cargar_parametros()
ruta_datos = PROCESSED_DATA_DIR / "estacion_hora_v3.parquet"

datos = pd.read_parquet(ruta_datos)
print(f"Dataset cargado: {datos.shape[0]:,} filas x {datos.shape[1]} columnas")
datos.head(3)

Dataset cargado: 3,445,646 filas x 21 columnas


,estacion_id,fecha_hora,hora,dia_semana,mes,fin_semana,salidas_lag_1,llegadas_lag_1,salidas_lag_24,llegadas_lag_24,...,llegadas_lag_168,festivo,hora_sin,hora_cos,dia_semana_sin,dia_semana_cos,mes_sin,mes_cos,fecha_objetivo,salidas_siguiente_hora
0,2,2025-10-01 06:00:00,7.0,2.0,10.0,0.0,2.0,2.0,NaN,NaN,...,NaN,0.0,0.965926,-0.258819,0.974928,-0.222521,-1.0,-1.836970e-16,2025-10-01 07:00:00,5
1,2,2025-10-01 07:00:00,8.0,2.0,10.0,0.0,5.0,4.0,NaN,NaN,...,NaN,0.0,0.866025,-0.500000,0.974928,-0.222521,-1.0,-1.836970e-16,2025-10-01 08:00:00,4
2,2,2025-10-01 08:00:00,9.0,2.0,10.0,0.0,4.0,10.0,NaN,NaN,...,NaN,0.0,0.707107,-0.707107,0.974928,-0.222521,-1.0,-1.836970e-16,2025-10-01 09:00:00,5


## 2. Entrenamiento y Evaluación Modular (Walk-Forward)

In [3]:
alpha_inicial = params["modelado"]["hiperparametros"]["ridge"]["alpha"][1]
estimador = Ridge(alpha=alpha_inicial)

# Ejecuta la validación temporal y registra el run en MLflow
modelo_final, tabla_ventanas, metricas_resumen = entrenar_y_evaluar(
    datos=datos,
    estimador=estimador,
    params=params,
    registrar=True,
    nombre_modelo="ridge_baseline",
)

print("Resultados por ventana walk-forward:")
display(tabla_ventanas)
print("Métricas promedio:", metricas_resumen)

2026-11-05 12:00:31.960 | INFO     | mibici.modeling.train:entrenar_y_evaluar:50 - Resultados por ventana:
2026-11-05 12:00:31.971 | INFO     | mibici.modeling.train:entrenar_y_evaluar:51 - 
 ventana mes_prueba  n_entrenamiento  n_prueba      mae     rmse     wape
       1    2026-04          1607955    265680 0.882540 1.516920 0.684965
       2    2026-05          1873635    275396 0.814383 1.344631 0.734055
       3    2026-06          2149019    288305 0.848375 1.467762 0.717018
       4    2026-07          2437278    326478 0.797560 1.376177 0.697918
       5    2026-08          2763730    345282 0.760776 1.318421 0.694984
       6    2026-09          3108998    336181 0.886527 1.573109 0.632115
2026-11-05 12:00:31.971 | INFO     | mibici.modeling.train:entrenar_y_evaluar:52 - Métricas promedio: {'mae_media': 0.8316933349948671, 'mae_desv': 0.045395280849530616, 'rmse_media': 1.4328365557990743, 'rmse_desv': 0.0931376328080098, 'wape_media': 0.6935091557474159, 'wape_desv': 0.03178

/opt/anaconda3/envs/mlops/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/05 12:00:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary

Resultados por ventana walk-forward:


,ventana,mes_prueba,n_entrenamiento,n_prueba,mae,rmse,wape
0,1,2026-04,1607955,265680,0.882540,1.516920,0.684965
1,2,2026-05,1873635,275396,0.814383,1.344631,0.734055
2,3,2026-06,2149019,288305,0.848375,1.467762,0.717018
3,4,2026-07,2437278,326478,0.797560,1.376177,0.697918
4,5,2026-08,2763730,345282,0.760776,1.318421,0.694984
5,6,2026-09,3108998,336181,0.886527,1.573109,0.632115


Métricas promedio: {'mae_media': 0.8316933349948671, 'mae_desv': 0.045395280849530616, 'rmse_media': 1.4328365557990743, 'rmse_desv': 0.0931376328080098, 'wape_media': 0.6935091557474159, 'wape_desv': 0.03178164222533578}


## 3. Registro del Modelo en el Model Registry de MLflow

In [4]:
# Obtener el último run principal (padre) registrado
cliente = mlflow.tracking.MlflowClient()
exp = cliente.get_experiment_by_name(params["mlflow"]["experimento"])

ultimo_run = cliente.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="attributes.run_name = 'ridge_baseline'",
    order_by=["start_time DESC"],
    max_results=1,
)[0]

# Registrar en el catálogo con el alias champion
version_modelo = registrar_modelo(
    run_id=ultimo_run.info.run_id,
    hiperparametros={"algoritmo": "Ridge", "alpha": alpha_inicial},
    metricas=metricas_resumen,
    nombre=params["mlflow"]["modelo_registrado"],
    alias="champion",
)
print(f"Modelo registrado: {params['mlflow']['modelo_registrado']} (Versión {version_modelo.version}) con alias 'champion'")


Registered model 'mibici-demanda-salidas' already exists. Creating a new version of this model...
2026/11/05 12:00:38 WARNING mlflow.tracking._model_registry.fluent: Run with id 90a7f7be2ee64039bed65f956528893b has no artifacts at artifact path 'modelo', registering model based on models:/m-4e61cf9191524cc489118d604bfd0e89 instead


Modelo registrado: mibici-demanda-salidas (Versión 1) con alias 'champion'


Created version '1' of model 'mibici-demanda-salidas'.


## 4. Inferencia con `mibici.modeling.predict`

In [5]:
# Predecir sobre una muestra utilizando el modelo champion cargado desde el Registry
muestra = datos.sample(5, random_state=params["modelado"]["semilla"])
predicciones = predecir(muestra, alias="champion")

resultado = muestra[["estacion_id", "fecha_hora", "salidas_siguiente_hora"]].copy()
resultado["prediccion"] = predicciones
display(resultado)

2026-11-05 12:00:38.552 | INFO     | mibici.modeling.predict:cargar_modelo:30 - Cargando modelo desde MLflow Model Registry: models:/mibici-demanda-salidas@champion


,estacion_id,fecha_hora,salidas_siguiente_hora,prediccion
51215,8,2026-08-06 21:00:00,0,0.194830
3345398,458,2026-08-18 07:00:00,1,2.397483
541564,65,2026-08-07 02:00:00,0,0.313847
1976620,249,2026-05-30 07:00:00,2,0.946440
194100,26,2025-12-01 05:00:00,0,0.949462
